In [ ]:
# %% [Block 0 — Setup and Plotly template]
# Principle 5: define style once in a template; every chart inherits it.

import os
import numpy as np
import pandas as pd
import pyreadstat
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

# Shared palette (kept consistent across the course)
PALETTE = ['#7BB3B2', '#65A6BD', '#C997AF', '#B8B0D3', '#F4CF97', '#98B9A0', '#F6DECD']

# Build the template once
nso_template = go.layout.Template()
nso_template.layout = go.Layout(
    font=dict(family='Arial', size=13, color='#333'),
    title_font=dict(size=16, color='#222'),
    plot_bgcolor='white',
    paper_bgcolor='white',
    colorway=PALETTE,
    xaxis=dict(showgrid=False),
    yaxis=dict(showgrid=True, gridcolor='lightgray', gridwidth=0.5),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0),
    margin=dict(l=60, r=30, t=60, b=40),
)
pio.templates['nso'] = nso_template
pio.templates.default = 'nso'

# Plotly toolbar config (clean modebar, sensible PNG export)
PLOTLY_CONFIG = {
    'displaylogo': False,
    'modeBarButtonsToRemove': ['select2d', 'lasso2d', 'autoScale2d'],
    'toImageButtonOptions': {'format': 'png', 'width': 1200, 'height': 700, 'scale': 2},
}

In [ ]:

ROOT_PATH = r'D:\Rowsquared\0_nextcloud\rowsquared\projects\25-07-SADC-PY4AFRICA\03_SADC-PY4AFRICA_Workspace\02_data\04_south_africa\Census2022SampleSTATA'

# Load the three Census 2022 files (Persons ⋈ Households on QID where useful)
geography,  geo_meta    = pyreadstat.read_dta(os.path.join(ROOT_PATH, 'Census2022Geography.dta'), )
households, hh_meta     = pyreadstat.read_dta(os.path.join(ROOT_PATH, 'Census2022Households.dta'))
persons,    pp_meta     = pyreadstat.read_dta(os.path.join(ROOT_PATH, 'Census2022Persons.dta'))

for df, meta in zip([geography, households, persons], [geo_meta, hh_meta, pp_meta]):
    for col, labels in meta.variable_value_labels.items():
        df[col] = df[col].map(labels)

# Quick sanity check after loading:
# df.shape, df.head(), df.dtypes


# Make sure that following are of correct type! DERH_HHAGE,

In [ ]:
# %% [Block 1 — Read the data]
# Geography and Households are ONE row per household — read them fully.
# Persons is large (one row per person) — cap the read at 100,000 rows.
# Despite the files being Stata (.dta), we read everything with pandas.

DATA_DIR = r'D:\Rowsquared\0_nextcloud\rowsquared\projects\25-07-SADC-PY4AFRICA\03_SADC-PY4AFRICA_Workspace\02_data\04_south_africa\Census2022SampleSTATA'  # adjust to your local path

geo = pd.read_stata(f'{DATA_DIR}/Census2022Geography.dta')
hh = pd.read_stata(f'{DATA_DIR}/Census2022Households.dta')

# Limit Persons to the first 100k rows with a chunked reader (avoids loading the lot).
with pd.read_stata(f'{DATA_DIR}/Census2022Persons.dta', chunksize=100_000) as reader:
    persons = next(reader)


In [ ]:
#hh = households
#geo = geography
# --- Fix dtypes ---
# read_stata maps value labels to strings, so some genuinely numeric columns
# (age, household size, weights, year) arrive as categorical/object. Coerce them.
for c in ['DERH_HSIZE', 'DERH_HHAGE', 'HH_WGT']:
    hh[c] = pd.to_numeric(hh[c], errors='coerce')

for c in ['P04_AGE', 'P03_YEAR', 'P12B_YEARMOVED', 'PERS_WGT']:
    persons[c] = pd.to_numeric(persons[c], errors='coerce')

# --- Treat unlabelled sentinel codes as missing ---
persons.loc[persons['P12B_YEARMOVED'] == 8888, 'P12B_YEARMOVED'] = np.nan

print('Geography:', geo.shape)
print('Households:', hh.shape)
print('Persons (capped):', persons.shape)
hh.head(3)

In [ ]:
# %% [Task 1 — Age distribution with median reference]
# Principle 1: one trace (Histogram) + layout. Principle 4: the median line is layout chrome.

# ----- Step 1: Prepare -----
age = persons['P04_AGE'].dropna()
median_age = age.median()

# ----- Step 2: Plot -----
fig = go.Figure(
    go.Histogram(x=age, nbinsx=20, marker_color=PALETTE[0], name='Age')
)

fig.add_vline(
    x=median_age, line_dash='dash', line_color='red',
    annotation_text=f'Median: {median_age:.0f} years',
    annotation_position='top right',
)

fig.update_layout(
    title='Age distribution — South Africa Census 2022 (sample)',
    xaxis_title='Age (completed years)',
    yaxis_title='Number of people',
)
fig.show(config=PLOTLY_CONFIG)

In [ ]:
valid = persons[persons['P02_SEX'].isin(['Male', 'Female'])]
pyramid = (
    pd.crosstab(valid['AGE_GROUP'], valid['P02_SEX'])
#    .dropna(how='all')       # drop any bands that aren't present in the sample
)
pyramid


# ----- Step 2: Plot -----
fig = go.Figure()
fig.add_trace(go.Bar(
    y=pyramid.index, x=-pyramid['Male'], name='Male',      # men extend left (negative)
    orientation='h', marker_color=PALETTE[1],
    customdata=pyramid['Male'],
    hovertemplate='Age %{y}: %{customdata:,} men<extra></extra>',
))
fig.add_trace(go.Bar(
    y=pyramid.index, x=pyramid['Female'], name='Female',   # women extend right (positive)
    orientation='h', marker_color=PALETTE[2],
    hovertemplate='Age %{y}: %{x:,} women<extra></extra>',
))
fig.update_layout(
    title='Population pyramid — South Africa Census 2022 (sample)',
    xaxis_title='Number of people',
    yaxis_title='Age group',
    barmode='relative',
    bargap=0.05,
    height=600,
    xaxis=dict(
        #tickvals=np.concatenate([-ticks[1:][::-1], ticks]),
        #ticktext=[f'{t:,}' for t in ticks[1:][::-1]] + [f'{t:,}' for t in ticks],
    ),
)
fig.show(config=PLOTLY_CONFIG)

In [ ]:
ticks[1:][::-1]

In [ ]:
# %% [Task 1b — Population pyramid by age group and sex]
# Principle 2: one trace per series (Male / Female).
# Principle 4: barmode='relative' makes the two sides mirror into a pyramid,
# once we flip the male counts to negative.

# ----- Step 1: Prepare -----
# AGE_GROUP is already a clean 5-year band; list it youngest -> oldest.
age_order = [
    '0 - 4', '5 - 9', '10 - 14', '15 - 19', '20 - 24', '25 - 29',
    '30 - 34', '35 - 39', '40 - 44', '45 - 49', '50 - 54', '55 - 59',
    '60 - 64', '65 - 69', '70 - 74', '75 - 79', '80 - 84', '85+',
]

valid = persons[persons['P02_SEX'].isin(['Male', 'Female'])]
pyramid = (
    pd.crosstab(valid['AGE_GROUP'], valid['P02_SEX'])
    # .reindex(age_order)      # impose the natural age ordering
    .dropna(how='all')       # drop any bands that aren't present in the sample
)

# ----- Step 2: Plot -----
fig = go.Figure()
fig.add_trace(go.Bar(
    y=pyramid.index, x=-pyramid['Male'], name='Male',      # men extend left (negative)
    orientation='h', marker_color=PALETTE[1],
    customdata=pyramid['Male'],
    hovertemplate='Age %{y}: %{customdata:,} men<extra></extra>',
))
fig.add_trace(go.Bar(
    y=pyramid.index, x=pyramid['Female'], name='Female',   # women extend right (positive)
    orientation='h', marker_color=PALETTE[2],
    hovertemplate='Age %{y}: %{x:,} women<extra></extra>',
))

# Build symmetric ticks that show ABSOLUTE counts on both sides of zero.
peak = pyramid.max().max()
order_mag = 10 ** int(np.floor(np.log10(peak)))
step = order_mag * 2 if peak / order_mag >= 5 else order_mag
ticks = np.arange(0, peak + step, step).astype(int)

fig.update_layout(
    title='Population pyramid — South Africa Census 2022 (sample)',
    xaxis_title='Number of people',
    yaxis_title='Age group',
    barmode='relative',
    bargap=0.05,
    height=600,
    xaxis=dict(
        tickvals=np.concatenate([-ticks[1:][::-1], ticks]),
        ticktext=[f'{t:,}' for t in ticks[1:][::-1]] + [f'{t:,}' for t in ticks],
    ),
)
fig.show(config=PLOTLY_CONFIG)

In [ ]:
# %% [Task 2 — People by population group]
# Principle 2: one trace per visual series. Principle 5: prepare first, plot second.

# ----- Step 1: Prepare -----
pop = persons['P07A_POP_GROUP']
pop = pop[~pop.isin(['Unspecified', 'Not applicable'])].value_counts().sort_values()

# ----- Step 2: Plot -----
fig = go.Figure(
    go.Bar(
        y=pop.index, x=pop.values, orientation='h',
        marker_color=PALETTE[0],
        text=pop.values, textposition='outside',
    )
)

fig.update_layout(
    title='People by population group',
    xaxis_title='Number of people',
    height=max(400, len(pop) * 40),
)
fig.show(config=PLOTLY_CONFIG)

In [ ]:
# %% [Task 3 — Households by province]
# Principle 2 again. Several provinces, so we use a vertical bar with rotated labels.

# ----- Step 1: Prepare -----
province = geo['Province'].value_counts().sort_values(ascending=False)

# ----- Step 2: Plot -----
fig = go.Figure(
    go.Bar(x=province.index, y=province.values, marker_color=PALETTE[1])
)

fig.update_layout(
    title='Households interviewed by province',
    yaxis_title='Number of households',
    xaxis_tickangle=-45,
    height=500,
)
fig.show(config=PLOTLY_CONFIG)

In [ ]:
hh['DERH_HSIZE'].unique()

In [ ]:
# %% [Task 4 — Household size distribution (donut)]
# Principle 1: a pie is still one trace + layout. Use pies only with 2–4 categories,
# so we bucket the numeric household size into 4 bands first.

# ----- Step 1: Prepare -----
order = ['1 person', '2–3', '4–5', '6+']
bands = pd.cut(
    hh['DERH_HSIZE'],
    bins=[0, 1, 3, 5, np.inf],
    labels=order,
)
size_counts = bands.value_counts().reindex(order)

# ----- Step 2: Plot -----
fig = go.Figure(
    go.Pie(
        labels=size_counts.index, values=size_counts.values, hole=0.4,
        marker=dict(colors=PALETTE[:4], line=dict(color='white', width=1.5)),
        textinfo='percent+label',
        sort=False
    )
)

fig.update_layout(title='Household size distribution')
fig.show(config=PLOTLY_CONFIG)

In [ ]:
hh['H03_TENURE'].unique()

In [ ]:
# %% [Task 5 — Household landscape, 2x2 subplots]
# Principle 3: several things in one figure = several traces, via make_subplots.

# ----- Step 1: Prepare -----
tenure = hh['H03_TENURE']
tenure = tenure[~tenure.isin(['Unspecified', 'Not applicable'])].value_counts().sort_values()

dwelling_top = hh['H02_MAINDWELLING'].value_counts().head(6).sort_values()
popgrp = hh['DERH_HHPOP'].value_counts().sort_values()

internet = hh['H13_INTERNET_ACCESS']
internet = internet[~internet.isin(['Unspecified'])].value_counts()

# ----- Step 2: Plot -----
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Tenure status', 'Top dwelling types',
                    'Head population group', 'Internet access'),
    specs=[[{'type': 'bar'}, {'type': 'bar'}],
           [{'type': 'bar'}, {'type': 'pie'}]],
)

fig.add_trace(
    go.Bar(y=tenure.index, x=tenure.values, orientation='h',
           marker_color=PALETTE[0], showlegend=False),
    row=1, col=1,
)
fig.add_trace(
    go.Bar(y=dwelling_top.index, x=dwelling_top.values, orientation='h',
           marker_color=PALETTE[1], showlegend=False),
    row=1, col=2,
)
fig.add_trace(
    go.Bar(y=popgrp.index, x=popgrp.values, orientation='h',
           marker_color=PALETTE[2], showlegend=False),
    row=2, col=1,
)
fig.add_trace(
    go.Pie(labels=internet.index, values=internet.values,
           marker=dict(colors=PALETTE), textinfo='percent', showlegend=True),
    row=2, col=2,
)

fig.update_layout(title='South African households at a glance', height=850)
fig.show(config=PLOTLY_CONFIG)

In [ ]:
# %% [Task 6 — Household head age: distribution + box by sex]
# Principle 3: mixed chart types in one figure. Principle 5: prepare first, plot second.

# ----- Step 1: Prepare -----
head_age = hh['DERH_HHAGE'].dropna()
median_age = head_age.median()

# ----- Step 2: Plot -----
fig = make_subplots(rows=1, cols=2, subplot_titles=('Distribution', 'By head sex'))

fig.add_trace(
    go.Histogram(x=head_age, nbinsx=30, marker_color=PALETTE[0],
                 name='Head age', showlegend=False),
    row=1, col=1,
)

for i, sex in enumerate(['Female', 'Male']):
    subset = hh.loc[hh['DERH_HHSEX'] == sex, 'DERH_HHAGE'].dropna()
    fig.add_trace(go.Box(y=subset, name=sex, marker_color=PALETTE[i]), row=1, col=2)

fig.add_vline(
    x=median_age, line_dash='dash', line_color='red',
    annotation_text=f'Median: {median_age:.0f}', row=1, col=1,
)

fig.update_layout(title='Age of the household head — distribution and by sex')
fig.update_xaxes(title_text='Age (years)', row=1, col=1)
fig.update_yaxes(title_text='Number of households', row=1, col=1)
fig.update_yaxes(title_text='Age (years)', row=1, col=2)
fig.show(config=PLOTLY_CONFIG)

In [ ]:
keep_tenure = ['Owned and fully paid off', 'Owned, but not yet paid off',
               'Rented from private individual', 'Occupied rent-free']
tmp = hh[hh['H03_TENURE'].isin(keep_tenure)]
comp = (
    pd.crosstab(tmp['DERH_HHPOP'], tmp['H03_TENURE'], normalize='index')

)
comp

In [ ]:
# %% [Task 7 — Tenure composition by population group, stacked bar]
# Principle 4: barmode='stack' lives in the layout.
# crosstab + normalize turns counts into shares that sum to 100% per group.
# No merge needed — tenure and population group both live in the Households file.

# ----- Step 1: Prepare -----
keep_tenure = ['Owned and fully paid off', 'Owned, but not yet paid off',
               'Rented from private individual', 'Occupied rent-free']
tmp = hh[hh['H03_TENURE'].isin(keep_tenure)]
comp = (
    pd.crosstab(tmp['DERH_HHPOP'], tmp['H03_TENURE'], normalize='index')
    .mul(100)[keep_tenure]
)

# ----- Step 2: Plot -----
fig = go.Figure()
for col, color in zip(keep_tenure, PALETTE[:4]):
    fig.add_trace(
        go.Bar(y=comp.index, x=comp[col], name=col,
               orientation='h', marker_color=color)
    )

fig.update_layout(
    title='Tenure composition by population group (%)',
    xaxis_title='Share of households (%)',
    barmode='stack', height=450,
)
fig.show(config=PLOTLY_CONFIG)

In [ ]:
# %% [Task 8 — Basic-services heatmap by province] (advanced — uses a merge)
# Principle 1: one Heatmap trace + layout. Principle 5: all the work is in the prep.
# Both files are one row per household, so this is a safe one-to-one merge on QID.

# ----- Step 1: Prepare -----
hh_geo = hh.merge(geo[['QID', 'Province']], on='QID', how='left')

# Build binary access flags from the labelled categorical columns.
hh_geo['piped_inside'] = hh_geo['H05_WATERPIPED'].eq('Piped (tap) water inside the dwelling')
hh_geo['flush_toilet'] = hh_geo['H08_TOILET'].str.startswith('Flush toilet', na=False)
hh_geo['electric_light'] = hh_geo['H10_ENERGY_LIGHTING'].eq('Electricity from mains')
hh_geo['refuse_collected'] = hh_geo['H11_REFUSE'].str.startswith('Removed by local authority', na=False)
hh_geo['has_internet'] = ~hh_geo['H13_INTERNET_ACCESS'].isin(
    ['No access to internet services', 'Unspecified']
)

flag_cols = ['piped_inside', 'flush_toilet', 'electric_light', 'refuse_collected', 'has_internet']
flag_labels = ['Piped water inside', 'Flush toilet', 'Electric lighting',
               'Refuse collected', 'Internet access']

heatmap_data = (
    hh_geo.groupby('Province')[flag_cols].mean().mul(100)
    .rename(columns=dict(zip(flag_cols, flag_labels)))
)

# ----- Step 2: Plot -----
fig = go.Figure(
    go.Heatmap(
        z=heatmap_data.values,
        x=heatmap_data.columns.tolist(),
        y=heatmap_data.index.tolist(),
        text=heatmap_data.round(0).values,
        texttemplate='%{text:.0f}%',
        colorscale='YlGnBu',
        colorbar=dict(title='% of<br>households'),
    )
)

fig.update_layout(
    title='Access to basic services by province (% of households)',
    height=500,
)
fig.show(config=PLOTLY_CONFIG)

In [ ]:
hunger_levels = ['Never', 'Seldom', 'Sometimes', 'Often', 'Always']
pct = (
    hh['A4_ADULT_HUNGER']
    .where(hh['A4_ADULT_HUNGER'].isin(hunger_levels))
    .value_counts(normalize=True)
    .reindex(hunger_levels, fill_value=0)
    .mul(100)
)
pct

In [ ]:
# %% [Task 9 — Diverging bar: frequency of adult hunger]
# Principle 2: one trace per Likert level.
# Principle 4: barmode='relative' is the layout setting that creates the diverging effect.

# ----- Step 1: Prepare -----
hunger_levels = ['Never', 'Seldom', 'Sometimes', 'Often', 'Always']
pct = (
    hh['A4_ADULT_HUNGER']
    .where(hh['A4_ADULT_HUNGER'].isin(hunger_levels))
    .value_counts(normalize=True)
    .reindex(hunger_levels, fill_value=0)
    .mul(100)
)

# Cool for 'Never'/'Seldom', neutral for 'Sometimes', hot for 'Often'/'Always'
diverging_colors = ['#4575b4', '#91bfdb', '#fee090', '#fc8d59', '#d73027']

# ----- Step 2: Plot -----
fig = go.Figure()
for i, (label, value, color) in enumerate(zip(hunger_levels, pct, diverging_colors)):
    # Levels 1–2 pull left (negative x), levels 3–5 pull right (positive x)
    x_val = -value if i < 2 else value
    fig.add_trace(
        go.Bar(
            y=[' '], x=[x_val], name=label, orientation='h', marker_color=color,
            text=[f'{value:.0f}%' if value >= 4 else ''],
            textposition='inside', textfont=dict(color='white'),
        )
    )

fig.add_vline(x=0, line_color='gray', line_width=1)
fig.update_layout(
    title='How often did adults experience hunger? (% of households)',
    xaxis_title='Share of households',
    barmode='relative', height=250,
)
fig.show(config=PLOTLY_CONFIG)

# Homework: replicate this for A5_CHILD_HUNGER (drop the
# 'Not applicable (no child in the household)' responses first).

In [ ]:
# %% [Task 10 — Internet access by population group, grouped bar]
# Principle 2: one trace per group.

# ----- Step 1: Prepare -----
hh['has_internet'] = ~hh['H13_INTERNET_ACCESS'].isin(
    ['No access to internet services', 'Unspecified']
)
ct = (
    pd.crosstab(hh['DERH_HHPOP'], hh['has_internet'], normalize='index')
    .mul(100)
)
ct.columns = ['No internet' if c is False else 'Has internet' for c in ct.columns]

# ----- Step 2: Plot -----
fig = go.Figure()
for i, group in enumerate(['Has internet', 'No internet']):
    fig.add_trace(
        go.Bar(x=ct.index, y=ct[group], name=group, marker_color=PALETTE[i])
    )

fig.update_layout(
    title='Internet access by household head population group',
    yaxis_title='Share of households (%)',
    barmode='group', xaxis_tickangle=-20, height=500,
)
fig.show(config=PLOTLY_CONFIG)

In [ ]:
# %% [Task 11 — Ownership of household assets, stacked Yes/No]
# Principle 2 + Principle 5: same recipe applied to several asset columns.

# ----- Step 1: Prepare -----
asset_cols = {
    'H12_CELLPHONE': 'Cellphone',
    'H12_TELEVISION': 'Television',
    'H12_REFRIGERATOR': 'Refrigerator',
    'H12_WASHINGM': 'Washing machine',
    'H12_MOTOR_CAR': 'Motor car',
    'H12_COMPUTER': 'Computer',
}

rows = []
for col, label in asset_cols.items():
    s = hh[col]
    s = s[s.isin(['Yes', 'No'])]  # ignore 'Unspecified'
    total = len(s)
    rows.append({
        'asset': label,
        'Yes': (s == 'Yes').sum() / total * 100,
        'No': (s == 'No').sum() / total * 100,
    })

plot_df = pd.DataFrame(rows).set_index('asset').sort_values('Yes')

# ----- Step 2: Plot -----
fig = go.Figure()
fig.add_trace(
    go.Bar(
        y=plot_df.index, x=plot_df['Yes'], name='Owns', orientation='h',
        marker_color=PALETTE[0],
        text=[f'{v:.0f}%' for v in plot_df['Yes']],
        textposition='inside', textfont=dict(color='white'),
    )
)
fig.add_trace(
    go.Bar(
        y=plot_df.index, x=plot_df['No'], name='Does not own', orientation='h',
        marker_color='#D3D3D3',
    )
)

fig.update_layout(
    title='Ownership of household assets',
    xaxis_title='Share of households (%)', xaxis_range=[0, 100],
    barmode='stack', height=400,
)
fig.show(config=PLOTLY_CONFIG)

In [ ]:
# %% [Task 12 — Household size vs head age, scatter by population group]
# Principle 1: one Scatter trace per group. Both axes are linear here — unlike sales,
# household size and age don't span orders of magnitude, so no log transform is needed.

# ----- Step 1: Prepare -----
plot_df = hh[['DERH_HSIZE', 'DERH_HHAGE', 'DERH_HHPOP']].sample(10000).dropna()
plot_df = plot_df[(plot_df['DERH_HSIZE'] > 0) & (plot_df['DERH_HHAGE'] > 0)]
groups = ['Black African', 'Coloured', 'White', 'Indian/Asian']

# ----- Step 2: Plot -----
# One trace per population group so the legend distinguishes them.
fig = go.Figure()
for i, grp in enumerate(groups):
    subset = plot_df[plot_df['DERH_HHPOP'] == grp]
    fig.add_trace(
        go.Scatter(
            x=subset['DERH_HHAGE'], y=subset['DERH_HSIZE'],
            mode='markers', name=grp,
            marker=dict(color=PALETTE[i], size=6, opacity=0.4),
        )
    )

fig.update_layout(
    title='Household size vs age of household head',
    xaxis_title='Age of household head (years)',
    yaxis_title='Household size (people)',
)
fig.write_html('fig.html')

In [ ]:
import os
os.getcwd()

In [ ]:
# %% [Capstone (optional) — services heatmap + hunger Likert in one figure]
# Nothing new conceptually: make_subplots + add_trace(..., row=, col=).
# Reuses heatmap_data from Task 8 and pct from Task 9.

fig = make_subplots(
    rows=1, cols=2,
    column_widths=[0.7, 0.3],
    subplot_titles=('Access to services by province', 'Adult hunger — % of households'),
    specs=[[{'type': 'heatmap'}, {'type': 'bar'}]],
)

# Left: heatmap (reuse heatmap_data from Task 8)
fig.add_trace(
    go.Heatmap(
        z=heatmap_data.values,
        x=heatmap_data.columns.tolist(),
        y=heatmap_data.index.tolist(),
        colorscale='YlGnBu',
        colorbar=dict(title='%', x=0.62),
    ),
    row=1, col=1,
)

# Right: diverging Likert (reuse pct from Task 9)
for i, (label, value, color) in enumerate(zip(hunger_levels, pct, diverging_colors)):
    x_val = -value if i < 2 else value
    fig.add_trace(
        go.Bar(y=[' '], x=[x_val], name=label, orientation='h', marker_color=color),
        row=1, col=2,
    )

fig.update_layout(
    title='Living conditions in South Africa, Census 2022',
    barmode='relative', height=550,
)
fig.show(config=PLOTLY_CONFIG)

In [ ]:
# %% [Task 13 — Does asset wealth track with education?] (advanced — uses a merge)
# Goal: build a simple asset-count index per household, then relate it to a
# person-level education outcome. Many-to-one merge: each person inherits their
# household's asset index (Persons -> Households on QID).

# ----- Step 1: Prepare -----
# (a) Asset-count index: how many of the 12 durable assets the household owns (0–12).
asset_cols = [
    'H12_REFRIGERATOR', 'H12_ELECTRIC_GAS_STOVE', 'H12_VACUUM_CLEANER',
    'H12_WASHINGM', 'H12_COMPUTER', 'H12_SATELLITE', 'H12_DVD_PLAYER',
    'H12_MOTOR_CAR', 'H12_TELEVISION', 'H12_RADIO', 'H12_LANDLINE', 'H12_CELLPHONE',
]
hh['asset_index'] = (hh[asset_cols] == 'Yes').sum(axis=1)   # 'No'/'Unspecified' count as 0

# (b) Person-level outcome: completed matric or higher, among adults aged 25+.
tertiary_terms = 'Grade 12|Matric|Bachelors|Masters|Honours|Doctoral|Diploma|Degree|Higher'
adults = persons[persons['P04_AGE'] >= 25].copy()
adults['matric_plus'] = adults['P21_EDULEVEL'].str.contains(
    tertiary_terms, case=False, na=False
)

# (c) Attach each adult's household asset index, then summarise by index level.
adults = adults.merge(hh[['QID', 'asset_index']], on='QID', how='left')
summary = (
    adults.dropna(subset=['asset_index'])
    .groupby('asset_index')['matric_plus']
    .agg(['mean', 'size'])
)
summary['mean'] *= 100   # share -> %

# ----- Step 2: Plot -----
fig = go.Figure(
    go.Scatter(
        x=summary.index, y=summary['mean'],
        mode='lines+markers', line=dict(color=PALETTE[1], width=3),
        marker=dict(size=9, color=PALETTE[1]),
        customdata=summary['size'],
        hovertemplate='Assets owned: %{x}<br>%{y:.0f}% matric+<br>n = %{customdata:,}<extra></extra>',
    )
)

overall = adults['matric_plus'].mean() * 100
fig.add_hline(
    y=overall, line_dash='dash', line_color='gray',
    annotation_text=f'Overall: {overall:.0f}%', annotation_position='bottom right',
)

fig.update_layout(
    title='Education vs household asset wealth (adults 25+)',
    xaxis_title='Household asset-count index (0–12 durables owned)',
    yaxis_title='% who completed matric or higher',
    yaxis_range=[0, 100],
    height=500,
)
fig.show(config=PLOTLY_CONFIG)

# Interpretation: this is an association, not causation. The asset index is a crude
# wealth proxy, and 'Unspecified' assets are simply treated as 'not owned'.

In [ ]:
# %% [Task 14 — Lifetime internal-migration matrix] (province of birth -> usual residence)
# Goal: cross-tabulate province of BIRTH against province of USUAL RESIDENCE now.
# The diagonal = people still living in their province of birth ('stayers');
# every off-diagonal cell is an internal lifetime-migration flow.
# No merge needed — both columns live in the Persons file.

# ----- Step 1: Prepare -----
provinces = [
    'Western Cape', 'Eastern Cape', 'Northern Cape', 'Free State',
    'KwaZulu-Natal', 'North West', 'Gauteng', 'Mpumalanga', 'Limpopo',
]

# Keep only people born in (and now usually resident in) one of the 9 provinces.
# This also drops foreign-born and 'Not applicable' / 'Unspecified' rows.
mig = persons[
    persons['P11_PROV_POB'].isin(provinces)
    & persons['DERP_USUALRESPROV'].isin(provinces)
]

# Rows = province of birth, columns = province of usual residence.
matrix = (
    pd.crosstab(mig['P11_PROV_POB'], mig['DERP_USUALRESPROV'])
    .reindex(index=provinces, columns=provinces, fill_value=0)
)

# Row-normalise: of everyone born in province X, what % now live in province Y?
matrix_pct = matrix.div(matrix.sum(axis=1).replace(0, np.nan), axis=0).mul(100)

# ----- Step 2: Plot -----
fig = go.Figure(
    go.Heatmap(
        z=matrix_pct.values,
        x=matrix_pct.columns.tolist(),   # province of usual residence (now)
        y=matrix_pct.index.tolist(),     # province of birth
        text=matrix_pct.round(0).values,
        texttemplate='%{text:.0f}',
        colorscale='Blues',
        colorbar=dict(title='% of those<br>born in row'),
    )
)

fig.update_layout(
    title='Lifetime internal migration: province of birth → province of usual residence',
    xaxis_title='Province of usual residence (now)',
    yaxis_title='Province of birth',
    xaxis_tickangle=-30,
    height=600,
)
# A strong diagonal means most people still live where they were born; bright
# off-diagonal cells are the inter-provincial migration flows.
fig.show(config=PLOTLY_CONFIG)

# Homework: instead of row %, plot raw counts and mask the diagonal
# (np.fill_diagonal on a copy of matrix.values) to make the flows stand out.